# In Class Activity April 14th, 2026

In [28]:
#!pip install optuna

### Importing libraries, preparing data, initial EDA

In [29]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna
import warnings
warnings.filterwarnings("ignore")

In [30]:
# importing data
adult = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [31]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


python(33729) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





- Many of the variables are very skewed. The age data has a lot less poeple ages 60+. Most people work in the private sector, and there aren't many people who are marked as separated or other in marital-status. Occupation also is overwhelmingly other, while over 60% of people were male. Income was also heavily imbalanced, with seemingly 70% of people making 50K or less. Almost all people in the dataset are white.

- There is a somewhat strong association between education and occupation, as well as with marital status and age.

- These imbalances will affect modeling, especially the income variable since it is the target. Accuracy alone may not be a good metric, so metrics like precision, recall, or F1-score will be more useful. I may also need to handle imbalance with techniques like class weights or resampling. These associations and others can reveal possibly redundant features, but can also be useful for prediction.

### Data Preprocessing (minimal) and Baseline Model

In [32]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [33]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [34]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

- The  XGBoost model achieved a mean F1 score of about 0.712 using 5-fold stratified CV. The scores are consistent across all the folds which indicates stable performance

- The dataset is imbalanced, so the F1-score is a better metric than accuracy as it balances precision and recall for the minority class

- To improve performance, I will focus on:

    - Finding the best set of hyperparameters by testing different values, I'll probably try using GridSearchCV since I'm most familiar with it
    - Hyperparameter tuning using Optuna
    - Handling class imbalance
    - Feature engineering and removing redundant features where possible
    - Try to better handle missing values in the data

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [35]:
#your code here

#what each of my hyperparameters do
#max_depth: controls the maximum depth of the trees. Deeper trees can capture more complex patterns but may lead to overfitting.
#learning_rate: controls the step size at each iteration while moving toward a minimum of the loss function. 
#               a smaller learning rate makes the model more robust to overfitting but requires more trees to converge.
#scale_pos_weight: controls the balance of positive and negative weights, useful for unbalanced classes. gives more weight to the minority class

model1 = XGBClassifier(enable_categorical=True, max_depth=3, learning_rate=0.1, scale_pos_weight=1, random_state=42)
model2 = XGBClassifier(enable_categorical=True, max_depth=3, learning_rate=0.01, scale_pos_weight=1, random_state=42)
model3 = XGBClassifier(enable_categorical=True, max_depth=3, learning_rate=0.001, scale_pos_weight=3, random_state=42)
model4 = XGBClassifier(enable_categorical=True, max_depth=6, learning_rate=0.1, scale_pos_weight=1, random_state=42)
model5 = XGBClassifier(enable_categorical=True, max_depth=6, learning_rate=0.01, scale_pos_weight=1, random_state=42)
model6 = XGBClassifier(enable_categorical=True, max_depth=6, learning_rate=0.001, scale_pos_weight=3, random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, model in enumerate([model1, model2, model3, model4, model5, model6], 1):
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1')
    print(f"Model {i} F1: {scores.mean():.4f}")

print("Among the six models tested, Model 4 performed the best with a mean F1 score of 0.7130 using 5-fold stratified cross-validation.\nThis model had a max_depth of 6, a learning_rate of 0.1, and a scale_pos_weight of 1.")
print("This suggests that deeper trees improved performance while class weighting did not help here.")

Model 1 F1: 0.6889
Model 2 F1: 0.5612
Model 3 F1: 0.6620
Model 4 F1: 0.7130
Model 5 F1: 0.5904
Model 6 F1: 0.6833
Among the six models tested, Model 4 performed the best with a mean F1 score of 0.7130 using 5-fold stratified cross-validation.
This model had a max_depth of 6, a learning_rate of 0.1, and a scale_pos_weight of 1.
This suggests that deeper trees improved performance while class weighting did not help here.


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [36]:
# your code here
#grid of values I want GridSearchCV to search through for the best hyperparameters
param_grid = {
    "max_depth": [3, 6, 9],
    "learning_rate": [0.1, 0.01, 0.001],
    "scale_pos_weight": [1, 3, 6],
    "n_estimators": [100, 200],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
#explanation of new hyperparameters:
#n_estimators: controls the number of trees in the model. More trees can improve performance but also increase training time and risk of overfitting.
#subsample: controls the fraction of samples used for training each tree. A value less than 1.0 can help prevent overfitting by introducing randomness.
#colsample_bytree: controls the fraction of features used for training each tree. A value less than 1.0 can help prevent overfitting by introducing randomness in feature selection.

#building xgboost model and tuning hyperparameters with GridSearchCV
xgb = XGBClassifier(enable_categorical=True, random_state=42)

#using stratified k-fold CV to evaluate each combo in grid
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#gridsearchCV to find best F1 Score
grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)

#fitting grid search to data
grid.fit(X, y)

#printing the best combo of hyperparameters and the best F1 score associated with the combo
print("Best parameters:", grid.best_params_)
print("Best F1 score:", grid.best_score_)

python(33732) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33733) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33734) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33735) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33736) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33737) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33738) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(33739) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 200, 'scale_pos_weight': 3, 'subsample': 0.8}
Best F1 score: 0.7198778114330251


In [37]:
#final model with best hyperparameters from GridSearchCV
final_model = XGBClassifier(
    enable_categorical=True,
    random_state=42,
    colsample_bytree=1.0,
    learning_rate=0.1,
    max_depth=9,
    n_estimators=200,
    scale_pos_weight=3,
    subsample=0.8
)

final_model.fit(X_train, y_train)

#predictions on test set
y_pred = final_model.predict(X_test)

#model performance
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test F1 Score:", f1_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.8475790766711024
Test F1 Score: 0.7164349647686155

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.86      0.90      7431
           1       0.65      0.80      0.72      2338

    accuracy                           0.85      9769
   macro avg       0.79      0.83      0.81      9769
weighted avg       0.86      0.85      0.85      9769



- After tuning with GridSearchCV and 5-fold stratified k fold, the best parameter combination included max_depth=9, learning_rate=0.1, scale_pos_weight=3, n_estimators=200, subsample=0.8, and colsample_bytree=1.0, achieving a cross-validated F1 score of 0.7199.

- The final model achieved an accuracy of 0.848 and an F1 score of 0.716. The model shows strong performance on the majority class, while maintaining a reasonable balance of precision (0.65) and recall (0.80) for the minority class.

### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [38]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10),
    "n_estimators": [100, 200],
    "subsample": [0.6, 0.8, 1.0]
}

# replace this placeholder model with your preferred model from above

"""  xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')"""   

# preferred model from above
xgb = XGBClassifier(random_state=42, enable_categorical=True)

#RandomizedSearchCV
xgb_random = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=skf,
    scoring='f1',
    random_state=42,
    n_jobs=1
)

#fit search
xgb_random.fit(X_train, y_train)

#best results
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set


xgb_random_best = XGBClassifier(random_state=42, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'],
                                n_estimators=xgb_random.best_params_['n_estimators'],
                                subsample=xgb_random.best_params_['subsample'],
                                enable_categorical=True)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'subsample': 1.0, 'scale_pos_weight': np.float64(2.0), 'n_estimators': 100, 'max_depth': np.int64(7), 'learning_rate': np.float64(0.10666666666666666)}
Best F1 score from RandomizedSearchCV: 0.726101593789138
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.83      0.81      9769
weighted avg       0.87      0.86      0.86      9769



- I used RandomizedSearchCV to tune five hyperparameters of the XGBoost model using stratified cross-validation. The best model achieved a cross-validated F1 score of 0.726, slightly improving over my previous models

- On the test set, the model achieved about 0.86 accuracy and an F1 score of 0.73 for the minority class, with strong recall of 0.79. Overall, this model shows improved performance and better balance in predicting the minority class than before

### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [39]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, n_estimators = n_estimators, subsample = subsample, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'],
                                  n_estimators=study.best_params['n_estimators'],
                                  subsample=study.best_params['subsample'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 16:14:48,874] A new study created in memory with name: no-name-deebc86c-5a72-40fd-8d35-572dd135039a


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 16:14:52,495] Trial 0 finished with value: 0.7244457981186492 and parameters: {'scale_pos_weight': 2.3610447435260267, 'max_depth': 9, 'learning_rate': 0.09432640203604367, 'n_estimators': 168, 'subsample': 0.9367675723014626}. Best is trial 0 with value: 0.7244457981186492.
[I 2026-04-15 16:14:54,760] Trial 1 finished with value: 0.7142937124984108 and parameters: {'scale_pos_weight': 3.0841373405959054, 'max_depth': 7, 'learning_rate': 0.1647931847162583, 'n_estimators': 132, 'subsample': 0.6886935447562544}. Best is trial 0 with value: 0.7244457981186492.
[I 2026-04-15 16:14:57,637] Trial 2 finished with value: 0.7107575100754728 and parameters: {'scale_pos_weight': 3.9471294789788915, 'max_depth': 9, 'learning_rate': 0.09446087921292567, 'n_estimators': 144, 'subsample': 0.8719913547021658}. Best is trial 0 with value: 0.7244457981186492.
[I 2026-04-15 16:15:00,057] Trial 3 finished with value: 0.690179943022988 and parameters: {'scale_pos_weight': 8.606675694929665, 

Exception ignored in: <function ResourceTracker.__del__ at 0x105141bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102391bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106ebdbc0>
Traceback (most recent call last

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


- RandomizedSearchCV produced the best results, achieving the highest cross-validated F1 score (about 0.726) and slightly better test performance compared to GridSearchCV and Optuna

- GridSearchCV was straightforward, but computationally expensive due to evaluating all parameter combinations

- Optuna was more efficient and flexible, exploring the parameter space intelligently, but results were similar to the other methods

- Overall, RandomizedSearchCV is preferred for this task as it balances performance and efficiency without the heavy computation of GridSearch

**Ordering of Models by F1-Score**

1. RandomizedSearchCV (~0.726)
2. Optuna (~0.722)
3. GridSearchCV (~0.72)
4. Baseline/Manual Tuning (~0.713)